# The golden routing dataset

One row per labelled `(dataset, query_id)`: the three per-route objective
scores, the derived route, and the outcome shape. A label comes from running
`dense_only` / `pure_rrf` / `sparse_only` against the lane's indexed corpus
and scoring each ranking with `0.7·HitRate@1 + 0.3·NDCG@10` — never from
asking a model which route looks right. The build pipeline lives in
`notebooks/route_labels.ipynb`; this notebook only reads the artifact.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from composition import CellFill
from hybrid_search_rrf_dataset.labels import RouteLabels
from hybrid_search_rrf_dataset.objective import RouterObjective

selection = CellFill().build()
labels = RouteLabels(selection, objective=RouterObjective(min_relevance=1))
data = labels.load()
print(f"{len(data):,} labelled rows | {data['dataset'].nunique()} lanes | "
      f"{data['query_id'].nunique():,} distinct queries")
print(f"objective: {labels.objective.name}")

## 1 — What's in it

Shape decides what a row can teach. `routes_differ` carries the
dense-vs-sparse signal, `all_tied` says any route works, `all_zero` found
nothing relevant and stores no label.

In [ ]:
shapes = data["shape"].value_counts().rename_axis("shape").to_frame("rows")
shapes["share"] = (shapes["rows"] / len(data) * 100).round(1)
shapes

In [ ]:
differ = data[data["shape"] == "routes_differ"]
routes = differ["route"].value_counts().rename_axis("route").to_frame("wins")
routes["share of trainable"] = (routes["wins"] / len(differ) * 100).round(1)
routes

The corpora are why routing has something to learn: lanes differ in average
IDF and out-of-vocabulary share, and the best blind route moves with them.

In [ ]:
stats = pd.read_parquet("data/route_labels/lane_corpus_stats.parquet")
print(stats.groupby("target")["avg_idf"].agg(["count", "mean"]).round(3).to_string())
pd.concat([stats.nlargest(3, "avg_idf"), stats.nsmallest(3, "avg_idf")])[
    ["dataset", "avg_idf", "oov_share", "target"]
].round(3)

## 2 — One winner per query: the decisive core

The argmax form keeps a row for training only when one route clearly won.
Decisive means the winner put a relevant document at rank 1 and the
runner-up missed — under this objective exactly margin ≥ 0.4, derived from
the weights, not hand-picked. The bar is deliberately strict, and it keeps
about one row in nine. Twice the pre-repair count (the IDF and
selection-join fixes), but the other eight rows stay invisible to training.

In [ ]:
from hybrid_search_rrf_dataset.router import decisive_rows

before = pd.read_parquet(labels.labels_path.parent / "labels.backup-pre-idf-full.parquet")
now, then = decisive_rows(data), decisive_rows(before)
print(f"decisive now:        {len(now):,} of {len(data):,} rows")
print(f"decisive before fix: {len(then):,} of {len(before):,} rows")
now["winner"].value_counts().rename_axis("winner").to_frame("decisive wins")

## 3 — Three yes/no labels: ties become signal

SPEC d60 changes the label form, not the scores. Each route gets
`ok = score ≥ oracle − 0.3`, where 0.3 is the objective's own `ndcg_weight`
— the widest gap two routes can show while still sharing the same top-1
outcome. A tied row labels `[1, 1, 1]` and trains all three classifiers;
`all_zero` rows stay null. At tolerance 0 the view reproduces the stored
`route` column exactly, so nothing is overwritten — the same scores read a
second way.

In [ ]:
view = labels.acceptability().frame()
answerable = view[view["serve"].notna()]
print(f"decisive rows:      {len(now):,}")
print(f"acceptability rows: {len(answerable):,}  "
      f"({len(answerable) / len(now):.1f}x, zero new retrieval)")

pd.DataFrame({
    route: answerable[f"ok_{route}"].astype(bool).value_counts()
    for route in ("dense_only", "pure_rrf", "sparse_only")
}).T.rename(columns={True: "acceptable", False: "not acceptable"})

`serve` — the cheapest acceptable route — is the evaluation target, not a
training label. On tied rows it points at sparse, which is where the class
that argmax starved gets its training diet.

In [ ]:
serve = answerable["serve"].value_counts().rename_axis("serve").to_frame("rows")
serve["share"] = (serve["rows"] / len(answerable) * 100).round(1)
serve

## 4 — Growing the thin archetypes: augmentation

Half the archetype cells cannot fill their draw from natural supply
(`notebooks/selection_audit.ipynb` §3); those cells are generation targets. An
operator edits a real parent query under a declared meaning-preserving
transformation, so the child inherits the parent's human judgments (d43d).
`data/augmentation/pool.parquet` holds the accepted children.

In [ ]:
from composition.cells import CELLS_BY_NAME
from composition.compose import join_text
from dataset_registry import DATASETS

pool = pd.read_parquet("data/augmentation/pool.parquet")
for_cells = pool[pool["floor"].isin(CELLS_BY_NAME)]
print(f"{len(pool):,} accepted children | {len(for_cells):,} minted for "
      f"{for_cells['floor'].nunique()} archetype cells, the rest for "
      f"pre-cell band floors")

sample = for_cells.groupby("floor").head(1).head(6).copy()
as_parents = sample.drop(columns=["query_id"]).rename(
    columns={"parent_dataset": "dataset", "generated_from": "query_id"})
sample["parent"] = join_text(as_parents, {d.name: d for d in DATASETS}).values
sample.rename(columns={"floor": "cell"})[["cell", "operator", "parent"]].assign(
    parent=sample["parent"].str.slice(0, 70),
    child=sample["query"].str.slice(0, 70),
)

### One round, priced before it is paid for

`campaign.plan()` shows the whole pass — one row per hungry floor with the
operator it would use and the row count it targets — without an LLM call.
The commented line underneath runs one real round for a single floor and
appends its accepted children to the pool.

In [ ]:
from augmentation.campaign import AugmentationCampaign
from augmentation.config import AugmentationPaths
from augmentation.loop import AugmentationLoop
from augmentation.parents import ParentPool
from dataset_registry import DATASETS

paths = AugmentationPaths()
catalog = pd.read_parquet(paths.catalog).astype({"query_id": str})
parents = ParentPool(catalog, selection.astype({"query_id": str}),
                     {d.name: d for d in DATASETS})
loop = AugmentationLoop(selection, sheet_path=paths.cell_order_sheet, parents=parents)

plan = AugmentationCampaign(loop).plan()
plan.head(10)

In [ ]:
# one real round for the first plannable floor — spends LLM tokens:
# floor = plan[plan["action"] == "produce"].iloc[0]["floor"]
# loop.run(floor, n=3)